# 자연어 처리 1일차 (3) — WordPiece 단어 사전 만들기 : NSMC · `BertWordPieceTokenizer` · `BertTokenizerFast`

수업 필기 `단어사전` (Colab)을 **실행 가능하도록 정리 · 개선한 노트**입니다.
(교재 `자연어처리` 교재9 p.27~30 — WordPiece 학습 과정 · `##` 표시 · 복원 규칙 · 장점 · WordPiece 실습)

- 문장 · 단어 · 형태소 토큰화와 OOV 문제는 [`nlp_preprocessing_tokenization.ipynb`](./nlp_preprocessing_tokenization.ipynb), BPE 원리는 [`bpe_subword_tokenization.ipynb`](./bpe_subword_tokenization.ipynb).
- 원본에서 고친 부분은 **📝 고친 부분** 으로 따로 적어 두었습니다.

| 단계 | 도구 | 결과 |
|---|---|---|
| ① 단어 사전 학습 | `tokenizers.BertWordPieceTokenizer` | `vocab.txt` (서브워드 3만 개) |
| ② 문장 토큰화 | `transformers.BertTokenizerFast` | 토큰 → id → BERT 입력(`input_ids` · `attention_mask`) |

In [1]:
# 그래프 한글 폰트 설정 (운영체제별)
import platform
import matplotlib.pyplot as plt

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

## 1. 데이터 — 네이버 영화 리뷰(NSMC)

In [2]:
import os, zipfile, urllib.request

# ratings_train.txt(14.6MB)는 저장소에 ratings_train.zip 으로 올려 두었다 → 처음 한 번 압축 풀기
# 둘 다 없으면 원본 저장소(github.com/e9t/nsmc)에서 내려받기
for name in ['ratings_train.txt', 'ratings_test.txt']:
    if os.path.exists(name):
        continue
    zip_name = name.replace('.txt', '.zip')
    if os.path.exists(zip_name):
        zipfile.ZipFile(zip_name).extractall('.')
    else:
        urllib.request.urlretrieve('https://raw.githubusercontent.com/e9t/nsmc/master/' + name, name)
print({n: f'{os.path.getsize(n) / 1e6:.1f}MB' for n in ['ratings_train.txt', 'ratings_test.txt']})

{'ratings_train.txt': '14.6MB', 'ratings_test.txt': '4.9MB'}


In [3]:
import re
import pandas as pd
from collections import Counter

train = pd.read_csv('ratings_train.txt', sep='\t')     # 탭 구분 : id · document · label
test = pd.read_csv('ratings_test.txt', sep='\t')
print(f'데이터 로드 완료! train {len(train):,}행 · test {len(test):,}행')
print('리뷰 결측 :', train['document'].isna().sum() + test['document'].isna().sum(), '개')
test.head()

데이터 로드 완료! train 150,000행 · test 50,000행
리뷰 결측 : 8 개


,id,document,label
0,6270596,굳 ㅋ,1
1,9274899,GDNTOPCLASSINTHECLUB,0
2,8544678,뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아,0
3,6825595,지루하지는 않은데 완전 막장임... 돈주고 보기에는....,0
4,6723715,3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??,0


> **📝 고친 부분**
> - `from google.colab import drive` · `drive.mount(...)` · `/content/drive/Othercomputers/...` 경로 → **노트북과 같은 폴더**의 `ratings_*.txt` (Colab 은 왼쪽 📁 에 업로드)
> - `try / except` 로 오류를 출력만 하고 넘어가면 다음 셀에서 `df` 가 없어 다른 에러가 난다 → 파일이 없으면 바로 에러가 나게 그냥 읽음

## 2. 학습용 말뭉치 — 리뷰 문장만 뽑기

원본은 `ratings_test.txt` · `ratings_train.txt` 를 **그대로** 토크나이저에 넣었다. 이 파일들은 한 줄이 `id \t 리뷰 \t 라벨` 이라서 **리뷰 id 숫자(6270596 …)와 라벨, 헤더 `id document label`** 까지 학습된다.

In [4]:
docs = pd.concat([train['document'], test['document']]).dropna()     # 리뷰 텍스트만, 결측 제외
with open('nsmc_corpus.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(docs))                                          # 한 줄 = 리뷰 하나
print(f'말뭉치 : 리뷰 {len(docs):,}개 → nsmc_corpus.txt')
print(open('nsmc_corpus.txt', encoding='utf-8').read(200))

말뭉치 : 리뷰 199,992개 → nsmc_corpus.txt
아 더빙.. 진짜 짜증나네요 목소리
흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나
너무재밓었다그래서보는것을추천한다
교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정
사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 던스트가 너무나도 이뻐보였다
막 걸음마 뗀 3세부터 초등학교 1학년생인 8살용영화.ㅋㅋㅋ...


## 3. ① 단어 사전 학습 — `BertWordPieceTokenizer` (교재 p.30)

- 말뭉치의 **모든 문자 + 특수 토큰**에서 시작해서 WordPiece 점수가 높은 쌍을 병합 → `vocab_size` 개가 될 때까지
- `lowercase=False` : 영어 대소문자 유지, `strip_accents=False` : 악센트 제거 끔 (**한글이 자모로 분해되지 않게** — 6절)

In [5]:
from tokenizers import BertWordPieceTokenizer

wp_tokenizer = BertWordPieceTokenizer(lowercase=False, strip_accents=False)
wp_tokenizer.train(
    files=['nsmc_corpus.txt'],
    vocab_size=30000,          # 단어 사전 크기
    min_frequency=2,           # 2번 이상 나온 쌍만 병합 (기본값)
    special_tokens=['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]'],
    show_progress=False,
)

vocab = wp_tokenizer.get_vocab()                       # {토큰: id} 딕셔너리
print('전체 단어 수 :', len(vocab))
print('단어 자료형 :', type(vocab))
by_id = sorted(vocab, key=vocab.get)
print('id 순서 앞부분 :', by_id[:12])
print('일부 단어 사전 예시 :', [(w, vocab[w]) for w in by_id[5000:5010]])

전체 단어 수 : 30000
단어 자료형 : <class 'dict'>
id 순서 앞부분 : ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]', '!', '"', '%', '&', "'", '(', ')']
일부 단어 사전 예시 : [('##부족', 5000), ('##대를', 5001), ('##엇다', 5002), ('##VD', 5003), ('지루했다', 5004), ('작품은', 5005), ('명작이', 5006), ('유치하다', 5007), ('무섭지도', 5008), ('잔인한', 5009)]


In [6]:
sub = [w for w in vocab if w.startswith('##')]
num = [w for w in vocab if re.fullmatch(r'(##)?\d+', w)]
print(f'## 서브워드(접미 조각) : {len(sub):,}개 ({len(sub) / len(vocab):.1%})')
print(f'숫자만으로 된 토큰     : {len(num):,}개')
print('특수 토큰 id :', {t: vocab[t] for t in ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]']})

## 서브워드(접미 조각) : 8,893개 (29.6%)
숫자만으로 된 토큰     : 154개
특수 토큰 id : {'[PAD]': 0, '[UNK]': 1, '[CLS]': 2, '[SEP]': 3, '[MASK]': 4}


### 비교 — 원본처럼 파일을 그대로 학습하면?

In [7]:
raw_tokenizer = BertWordPieceTokenizer(lowercase=False, strip_accents=False)
raw_tokenizer.train(files=['ratings_test.txt', 'ratings_train.txt'], vocab_size=30000, show_progress=False)
raw_vocab = raw_tokenizer.get_vocab()
raw_num = [w for w in raw_vocab if re.fullmatch(r'(##)?\d+', w)]
print(f'원본 방식 숫자 토큰 : {len(raw_num):,}개 ({len(raw_num) / len(raw_vocab):.1%})  예: {raw_num[:8]}')
print(f'리뷰만 학습 숫자 토큰 : {len(num):,}개 ({len(num) / len(vocab):.1%})')
print('헤더 단어가 사전에? ', {w: w in raw_vocab for w in ['id', 'document', 'label']})

# 수업 때 받은 vocab.txt (원본 방식) 도 확인
user_vocab = open('vocab.txt', encoding='utf-8').read().split()
user_num = [w for w in user_vocab if re.fullmatch(r'(##)?\d+', w)]
print(f'필기 vocab.txt : {len(user_vocab):,}줄 중 숫자 토큰 {len(user_num):,}개')

원본 방식 숫자 토큰 : 5,778개 (19.3%)  예: ['##820', '##1280', '936', '##6813', '279', '##828', '##1076', '10067']
리뷰만 학습 숫자 토큰 : 154개 (0.5%)
헤더 단어가 사전에?  {'id': False, 'document': False, 'label': False}
필기 vocab.txt : 30,000줄 중 숫자 토큰 5,778개


원본 방식은 사전의 **약 19%**(5,700여 칸)를 리뷰 id 숫자 조각(`##6266` · `5233` …)에 써 버린다 — 필기 출력의 `('##6266', 18571)` · `('5233', 28409)` 가 바로 그것. 리뷰 문장만 학습하면 그 자리가 실제 단어 조각으로 채워진다.

> **📝 고친 부분**
> - 원본 `files=[path1, path2]` (id · 라벨이 섞인 원본 파일) → **`document` 열만** 뽑은 `nsmc_corpus.txt`
> - 변수 이름 `BartToken` → `wp_tokenizer` — **BART** 는 다른 모델(페이스북)이고, 이것은 **BERT WordPiece** 토크나이저
> - `from enum import Flag` → 쓰지 않는 import 삭제

## 4. 단어 사전 저장 — `vocab.txt`

In [8]:
import os

save_path = './wordpiece_vocab'                 # 폴더 (원본 필기 vocab.txt 와 섞이지 않게)
os.makedirs(save_path, exist_ok=True)
print(wp_tokenizer.save_model(save_path))       # 단어 사전은 파일로 저장된다 : 한 줄 = 토큰 하나, 줄 번호 = id

with open(os.path.join(save_path, 'vocab.txt'), encoding='utf-8') as f:
    lines = f.read().split('\n')
print('앞 8줄 :', lines[:8])
print("'영화' 의 줄 번호 =", lines.index('영화'), '| vocab id =', vocab['영화'])

['./wordpiece_vocab/vocab.txt']
앞 8줄 : ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]', '!', '"', '%']
'영화' 의 줄 번호 = 1979 | vocab id = 1979


## 5. ② 한글 문장 토큰화 — `BertTokenizerFast` (교재 p.30 ③)

In [9]:
# Colab 에는 transformers 가 설치되어 있다 (로컬 : pip install transformers)
import warnings
warnings.filterwarnings('ignore', message='IProgress not found')   # 진행 막대 경고 숨기기
from transformers import BertTokenizerFast

bert_tokenizer = BertTokenizerFast.from_pretrained(
    save_path,
    do_lower_case=False,       # ← BertTokenizerFast 의 이름은 lowercase 가 아니라 do_lower_case
    strip_accents=False,
)

sample_text = '가는말이고아야 오는말이 곱다'          # 필기 예시 (띄어쓰기 · 맞춤법 그대로)
sample_fixed = '가는 말이 고와야 오는 말이 곱다'      # 바른 속담
for s in [sample_text, sample_fixed]:
    print('한글 샘플 :', s)
    print('토큰화 결과 :', bert_tokenizer.tokenize(s))
    print('=' * 60)

한글 샘플 : 가는말이고아야 오는말이 곱다
토큰화 결과 : ['가는', '##말이', '##고', '##아야', '오는', '##말이', '곱', '##다']
한글 샘플 : 가는 말이 고와야 오는 말이 곱다
토큰화 결과 : ['가는', '말이', '고', '##와', '##야', '오는', '말이', '곱', '##다']


- 필기 결과 `['가는', '##말이', '##고', '##아야', '오는', '##말이', '곱', '##다']` 와 같다 ✔
- 띄어쓰기를 안 하면 `##말이` 처럼 앞 단어에 붙은 조각이 되고, `고아야` 오타는 `##고 ##아야` 로 쪼개진다. 띄어 쓰면 `말이` 가 **독립 단어**로 나온다 → 같은 말도 띄어쓰기 · 맞춤법에 따라 토큰이 달라진다

### `lowercase=False` 는 무시된다

In [10]:
for kw in [dict(lowercase=False, strip_accents=False), dict(do_lower_case=False, strip_accents=False)]:
    t = BertTokenizerFast.from_pretrained(save_path, **kw)
    print(f'{str(kw):48s} → {t.tokenize("Marvel 영화 OST 최고")}  (do_lower_case={t.do_lower_case})')

{'lowercase': False, 'strip_accents': False}     → ['m', '##ar', '##ve', '##l', '영화', 'ost', '최고']  (do_lower_case=True)
{'do_lower_case': False, 'strip_accents': False} → ['M', '##ar', '##ve', '##l', '영화', 'OST', '최고']  (do_lower_case=False)


> **📝 고친 부분** — 원본 `from_pretrained(save_path, lowercase=False, ...)` : `BertTokenizerFast` 에는 `lowercase` 인자가 **없어서** 기본값 `do_lower_case=True` 로 영어가 소문자가 된다(`Marvel` → `m ##ar ...`). 사전은 대소문자를 구분해 만들었으니 **`do_lower_case=False`** 로 맞춰야 한다.

## 6. `strip_accents=False` 를 빼면? — 한글이 전부 `[UNK]`

In [11]:
import unicodedata

t_default = BertTokenizerFast.from_pretrained(save_path)          # 옵션 없이
print('옵션 없이 :', t_default.tokenize(sample_fixed))

# 악센트 제거 = 유니코드 NFD 로 분해한 뒤 결합 문자를 지우는 것 → 한글 음절이 자모로 쪼개진다
s = '곱다'
nfd = unicodedata.normalize('NFD', s)
print(f"'{s}' 2글자 → NFD {len(nfd)}글자 :", [hex(ord(c)) for c in nfd])

옵션 없이 : ['[UNK]', '[UNK]', '[UNK]', '[UNK]', '[UNK]', '[UNK]']
'곱다' 2글자 → NFD 5글자 : ['0x1100', '0x1169', '0x11b8', '0x1103', '0x1161']


- `do_lower_case=True`(기본)이면 `strip_accents` 도 따라서 켜지고, 한글 음절이 **초성 · 중성 · 종성 자모**로 분해된다 → 음절 단위로 만든 사전에 없어서 전부 `[UNK]`
- 교재 p.27 예시의 `'##ᆻ다'` 는 이렇게 분해된 **종성 ㅆ(ᆻ)** 이 남은 모습 — 그래서 학습 · 토큰화 모두 `strip_accents=False` 를 맞춰야 한다

## 7. 토큰 → id (`convert_tokens_to_ids`)

In [12]:
tokens = bert_tokenizer.tokenize(sample_text)
token_id = bert_tokenizer.convert_tokens_to_ids(tokens)        # 단어 사전과 대조해 고유 번호(id)로
print('토큰 :', tokens)
print('토큰화 ID :', token_id)
print('id → 토큰 :', bert_tokenizer.convert_ids_to_tokens(token_id))

토큰 : ['가는', '##말이', '##고', '##아야', '오는', '##말이', '곱', '##다']
토큰화 ID : [3272, 2824, 1018, 3032, 7646, 2824, 146, 1064]
id → 토큰 : ['가는', '##말이', '##고', '##아야', '오는', '##말이', '곱', '##다']


필기 출력 `[3611, 3010, 1073, 3259, 8803, 3010, 146, 1069]` 과 id 가 다른 것은 정상 — 학습 데이터(원본 파일 vs 리뷰만)가 달라 **사전의 순서가 다르기** 때문. id 는 사전마다 다르므로 **모델과 사전은 항상 짝**으로 저장한다.

> **📝 고친 부분** — 주석 `대좋해` → **대조해**, `트렌스포머` → **트랜스포머**, 뜻이 불분명한 `#단어 생성 -> 사 생성` 삭제

## 8. BERT 입력 형태 — `[CLS]` · `[SEP]` · `attention_mask` (교재 p.30)

In [13]:
enc = bert_tokenizer(sample_fixed)
print('input_ids      :', enc['input_ids'])
print('토큰           :', bert_tokenizer.convert_ids_to_tokens(enc['input_ids']))
print('token_type_ids :', enc['token_type_ids'])
print('attention_mask :', enc['attention_mask'])

input_ids      : [2, 3272, 2187, 142, 1047, 1035, 7646, 2187, 146, 1064, 3]
토큰           : ['[CLS]', '가는', '말이', '고', '##와', '##야', '오는', '말이', '곱', '##다', '[SEP]']
token_type_ids : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
attention_mask : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [14]:
# 여러 문장을 한 번에 : 길이를 맞추려고 [PAD] 를 채우고, attention_mask 0 으로 '무시' 표시
batch = bert_tokenizer(['정말 재밌다', '배우들 연기는 좋았는데 스토리가 너무 지루했다', '노잼'],
                       padding=True, truncation=True, max_length=12)
for ids, mask in zip(batch['input_ids'], batch['attention_mask']):
    print(f'{str(bert_tokenizer.convert_ids_to_tokens(ids)):95s} mask={mask}')

['[CLS]', '정말', '재밌다', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']                             mask=[1, 1, 1, 1, 0, 0, 0, 0]
['[CLS]', '배우들', '연기는', '좋았는데', '스토리가', '너무', '지루했다', '[SEP]']                                  mask=[1, 1, 1, 1, 1, 1, 1, 1]
['[CLS]', '노잼', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']                           mask=[1, 1, 1, 0, 0, 0, 0, 0]


## 9. `##` 의 의미와 복원 규칙 (교재 p.27~29)

| 유형 | 표시 | 예 |
|---|---|---|
| 독립적인 단어 토큰 | `##` 없음, 뒤에 `##` 조각 없음 | 시간, 모두 |
| 단어의 **시작** 조각 | `##` 없음, 뒤에 `##` 조각이 붙음 | 너, 함께, 눈부 |
| 단어의 **중간 / 끝** 조각 | `##` 붙음 | ##와, ##한, ##셨다 |

복원 : `##` 없는 토큰 → **한 칸 띄우고** 쓰기, `##` 있는 토큰 → `##` 지우고 **앞에 붙이기**.

![WordPiece 복원 애니메이션](./images/anim_03_wordpiece_restore.gif)

In [15]:
def restore(tokens):
    text = ''
    for t in tokens:
        if t.startswith('##'):
            text += t[2:]                                   # 앞 토큰에 바짝 붙임
        else:
            text += (' ' if text else '') + t               # 한 칸 띄우고 씀
    return text


def token_types(tokens):
    types = []
    for i, t in enumerate(tokens):
        nxt = tokens[i + 1] if i + 1 < len(tokens) else ''
        if t.startswith('##'):
            types.append('##조각')
        elif nxt.startswith('##'):
            types.append('시작 조각')
        else:
            types.append('독립 단어')
    return types


text = '너와 함께한 시간 모두 눈부셨다'                    # 교재 p.27 예시
toks = bert_tokenizer.tokenize(text)
print('토큰 :', toks)
print('유형 :', list(zip(toks, token_types(toks))))
print('직접 복원 :', restore(toks))
print('decode    :', bert_tokenizer.decode(bert_tokenizer.convert_tokens_to_ids(toks)))
print('원문과 같은가?', restore(toks) == text)

토큰 : ['너와', '함께한', '시간', '모두', '눈부', '##셨다']
유형 : [('너와', '독립 단어'), ('함께한', '독립 단어'), ('시간', '독립 단어'), ('모두', '독립 단어'), ('눈부', '시작 조각'), ('##셨다', '##조각')]
직접 복원 : 너와 함께한 시간 모두 눈부셨다
decode    : 너와 함께한 시간 모두 눈부셨다
원문과 같은가? True


## 10. OOV 비교 — 어절 사전 vs WordPiece 사전 (교재 p.29 장점)

같은 **3만 개** 사전으로 테스트 리뷰를 표현할 때 `[UNK]` 가 얼마나 생기는지.

In [16]:
train_docs = train['document'].dropna()
test_docs = test['document'].dropna()

word_counts = Counter(w for d in train_docs for w in d.split())
vocab_word = {w for w, _ in word_counts.most_common(30000)}         # 어절 사전 3만

test_words = [w for d in test_docs for w in d.split()]
word_unk = sum(w not in vocab_word for w in test_words) / len(test_words)

# WordPiece 사전은 test 도 포함해 학습했으므로, 공정하게 train 리뷰만으로 다시 학습해서 비교
with open('nsmc_train_only.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(train_docs))
wp_train = BertWordPieceTokenizer(lowercase=False, strip_accents=False)
wp_train.train(files=['nsmc_train_only.txt'], vocab_size=30000, show_progress=False)

enc_test = wp_train.encode_batch(test_docs.tolist(), add_special_tokens=False)
n_tok = sum(len(e.tokens) for e in enc_test)
n_unk = sum(e.tokens.count('[UNK]') for e in enc_test)
unk_words = Counter(w for d, e in zip(test_docs, enc_test) if '[UNK]' in e.tokens for w in d.split()
                    if '[UNK]' in wp_train.encode(w, add_special_tokens=False).tokens)

print(f'어절 사전 3만      : 테스트 어절의 {word_unk:.1%} 가 [UNK]')
print(f'WordPiece 사전 3만 : 테스트 토큰의 {n_unk / n_tok:.3%} 가 [UNK]  (리뷰당 평균 {n_tok / len(test_docs):.1f} 토큰)')
print('WordPiece 에서도 [UNK] 인 어절 예 :', [w for w, _ in unk_words.most_common(8)])

어절 사전 3만      : 테스트 어절의 36.1% 가 [UNK]
WordPiece 사전 3만 : 테스트 토큰의 1.322% 가 [UNK]  (리뷰당 평균 14.1 토큰)
WordPiece 에서도 [UNK] 인 어절 예 : ['왤케', '및', '웰메이드', '곧', '완죤', '뗄', '스타뎀', '레이첼']


In [17]:
for w in ['존잼', '갓띵작', '노잼각', '꿀잼보장', '재미없었어요']:
    print(f'{w:8s} 어절 사전 : {"있음" if w in vocab_word else "[UNK]"}  | WordPiece : {wp_train.encode(w, add_special_tokens=False).tokens}')

존잼       어절 사전 : 있음  | WordPiece : ['존잼']
갓띵작      어절 사전 : [UNK]  | WordPiece : ['[UNK]']
노잼각      어절 사전 : [UNK]  | WordPiece : ['노잼', '##각']
꿀잼보장     어절 사전 : [UNK]  | WordPiece : ['꿀잼', '##보', '##장']
재미없었어요   어절 사전 : 있음  | WordPiece : ['재미없었어요']


- 어절 사전은 테스트 어절의 **36%** 가 `[UNK]` 인데, WordPiece 는 토큰의 **1.3%** 뿐
- 신조어 `노잼각` 은 `노잼 + ##각`, `꿀잼보장` 은 `꿀잼 + ##보 + ##장` 처럼 아는 조각의 조합으로 표현된다 → **OOV 해결**(교재 p.29 ①), 사전 3만 개로 20만 리뷰를 커버 → **가벼운 용량**(②)
- 그런데 `갓띵작` · `및` · `곧` · `왤케` 는 여전히 `[UNK]` — 어절 안에 **초기 문자 사전에 없는 글자**(`띵` · `및` · `곧` · `왤`)가 하나라도 있으면 WordPiece 는 **어절 전체**를 `[UNK]` 로 바꾼다

### `[UNK]` 를 더 줄이기 — `limit_alphabet`

`BertWordPieceTokenizer.train()` 은 초기 문자 사전을 **가장 많이 나온 글자 1,000개**(`limit_alphabet=1000` 기본값)로 자른다. 영어(알파벳 26개)는 충분하지만 한글 음절은 수천 개라 드문 글자가 잘려 나간다.

In [18]:
n_chars = len({ch for d in train_docs for ch in d})
print(f'학습 리뷰에 나온 서로 다른 글자 수 : {n_chars:,}')

wp_big = BertWordPieceTokenizer(lowercase=False, strip_accents=False)
wp_big.train(files=['nsmc_train_only.txt'], vocab_size=30000, limit_alphabet=6000, show_progress=False)
enc_big = wp_big.encode_batch(test_docs.tolist(), add_special_tokens=False)
n_tok2 = sum(len(e.tokens) for e in enc_big)
n_unk2 = sum(e.tokens.count('[UNK]') for e in enc_big)
print(f'limit_alphabet=1000 (기본) : [UNK] {n_unk / n_tok:.3%}')
print(f'limit_alphabet=6000        : [UNK] {n_unk2 / n_tok2:.3%}')
for w in ['갓띵작', '왤케', '및', '곧']:
    print(f'{w:5s} → {wp_big.encode(w, add_special_tokens=False).tokens}')

학습 리뷰에 나온 서로 다른 글자 수 : 3,004


limit_alphabet=1000 (기본) : [UNK] 1.322%
limit_alphabet=6000        : [UNK] 0.043%
갓띵작   → ['갓', '##띵', '##작']
왤케    → ['왤케']
및     → ['및']
곧     → ['곧']


리뷰에 나온 글자는 **3,004개** — 1,000개로 자르면 나머지 2,000여 글자가 든 어절이 통째로 `[UNK]` 가 된다. `limit_alphabet=6000` 으로 모든 글자를 넣으면 `[UNK]` 가 **1.3% → 0.04%**, `갓띵작` 도 `갓 + ##띵 + ##작` 으로 읽힌다. (`train()` 은 여러 스레드로 돌아서 실행마다 id · 비율이 아주 조금 달라질 수 있다)

## 11. 사전 크기에 따라 달라지는 토큰

![사전 크기 애니메이션](./images/anim_04_vocab_size_tradeoff.gif)

In [19]:
rows = []
probe = '배우들 연기는 좋았는데 스토리가 너무 지루했다'
for size in [1000, 5000, 30000]:
    t = BertWordPieceTokenizer(lowercase=False, strip_accents=False)
    t.train(files=['nsmc_train_only.txt'], vocab_size=size, show_progress=False)
    sample_enc = t.encode_batch(test_docs.sample(5000, random_state=0).tolist(), add_special_tokens=False)
    avg = sum(len(e.tokens) for e in sample_enc) / len(sample_enc)
    rows.append((size, round(avg, 1), ' '.join(t.encode(probe, add_special_tokens=False).tokens)))
pd.DataFrame(rows, columns=['사전 크기', '리뷰당 평균 토큰 수', '예시 문장 토큰'])

,사전 크기,리뷰당 평균 토큰 수,예시 문장 토큰
0,1000,28.0,배 ##우 ##들 연 ##기 ##는 좋 ##았 ##는 ##데 스 ##토 ##리 ##...
1,5000,18.1,배우들 연기는 좋았는데 스토리가 너무 지루했다
2,30000,14.1,배우들 연기는 좋았는데 스토리가 너무 지루했다


사전이 작으면 단어가 잘게(글자 단위에 가깝게) 쪼개져 문장이 길어지고, 크면 통째 단어가 늘어 짧아진다 — 사전 크기는 **메모리 vs 문장 길이**의 절충이다.

## 📝 고친 부분 정리 (필기 `단어사전`)

| 원본 | 고친 것 |
|---|---|
| 드라이브 마운트 · `/content/drive/Othercomputers/...` | 같은 폴더의 `ratings_*.txt` |
| `try / except` 로 로드 오류 숨김 | 그냥 읽기 (없으면 바로 에러) |
| `from enum import Flag` | 삭제 (안 씀) |
| 변수 `BartToken` | `wp_tokenizer` (BART 아님, BERT WordPiece) |
| 원본 파일 그대로 학습 (id · 라벨 · 헤더 포함) | `document` 열만 뽑은 말뭉치 — 숫자 토큰 5,7xx → 1xx개 |
| `BertTokenizerFast(..., lowercase=False)` | **`do_lower_case=False`** (`lowercase` 는 무시됨) |
| 저장 경로에 바로 `vocab.txt` | `./wordpiece_vocab/` 폴더 |
| 주석 `대좋해` · `트렌스포머` · `#단어 생성 -> 사 생성` | 대조해 · 트랜스포머 · 삭제 |
| `convert_tokens_to_ids` 까지 | `[CLS]`/`[SEP]` · `attention_mask` · 패딩 · 복원 · OOV 비교 추가 |
| `limit_alphabet` 기본값(1,000자) | 한글은 음절이 많아 드문 글자가 든 어절이 통째로 `[UNK]` → `limit_alphabet` 을 늘려 비교 |